In [1]:
import os
import subprocess
import tensorflow as tf
from pathlib import Path

2026-02-19 11:46:11.588614: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-02-19 11:46:17.614184: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


In [2]:
# ============================================================
# Tool paths (macro-like constants)
# ============================================================
MLIR_OPT = "/LLVM-22.1.0-rc1-Linux-X64/bin/mlir-opt"
MLIR_TRANSLATE = "/LLVM-22.1.0-rc1-Linux-X64/bin/mlir-translate"
MLIR_OPT_PYTORCHSIM = "/riscv-llvm/bin/mlir-opt"
STABLEHLO_OPT = "/workspace/stablehlo/build/bin/stablehlo-opt"
STABLEHLO_TRANSLATE = "/workspace/stablehlo/build/bin/stablehlo-translate"

OUT_DIR = "out"
os.makedirs(OUT_DIR, exist_ok=True)
def run(cmd):
    print(">>", " ".join(cmd))
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.stdout:
        print(result.stdout)
    if result.stderr:
        print(result.stderr)
    result.check_returncode()

In [3]:
# ============================================================
# 0. Target TensorFlow function
# ============================================================
@tf.function(jit_compile=True)
def add_fn(x, y):
    return x + y

x = tf.constant([1.0, 2.0], dtype=tf.float32)
y = tf.constant([3.0, 4.0], dtype=tf.float32)


I0000 00:00:1771501580.352121   18364 pluggable_device_factory.cc:305] Could not identify NUMA node of platform MY_DEVICE ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
I0000 00:00:1771501580.352258   18364 pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:MY_DEVICE:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: FAKE_TPU_SIM_DEVICE, pci bus id: <undefined>)


In [4]:
# ============================================================
# 1. HLO
# ============================================================
ir = add_fn.experimental_get_compiler_ir(x, y)(stage="hlo")
print("=== HLO ===")
print(ir)


=== HLO ===
HloModule a_inference_add_fn_8__.9, entry_computation_layout={(f32[2]{0}, f32[2]{0})->f32[2]{0}}

ENTRY %a_inference_add_fn_8__.9 (arg0.1: f32[2], arg1.2: f32[2]) -> f32[2] {
  %arg0.1 = f32[2]{0} parameter(0), parameter_replication={false}, metadata={op_name="XLA_Args"}
  %reshape.3 = f32[2]{0} reshape(f32[2]{0} %arg0.1)
  %arg1.2 = f32[2]{0} parameter(1), parameter_replication={false}, metadata={op_name="XLA_Args"}
  %reshape.4 = f32[2]{0} reshape(f32[2]{0} %arg1.2)
  %add.5 = f32[2]{0} add(f32[2]{0} %reshape.3, f32[2]{0} %reshape.4), metadata={op_type="AddV2" op_name="add" source_file="/opt/conda/lib/python3.10/site-packages/tensorflow/python/framework/ops.py" source_line=1221}
  %reshape.6 = f32[2]{0} reshape(f32[2]{0} %add.5), metadata={op_name="XLA_Retvals"}
  %tuple.7 = (f32[2]{0}) tuple(f32[2]{0} %reshape.6), metadata={op_name="XLA_Retvals"}
  ROOT %get-tuple-element.8 = f32[2]{0} get-tuple-element((f32[2]{0}) %tuple.7), index=0, metadata={op_name="XLA_Retvals"}
}



2026-02-19 11:46:20.490748: I external/local_xla/xla/service/service.cc:163] XLA service 0x1ba34e30 initialized for platform Host (this does not guarantee that XLA will be used). Devices:
2026-02-19 11:46:20.490806: I external/local_xla/xla/service/service.cc:171]   StreamExecutor device (0): Host, Default Version


In [5]:
# ============================================================
# 2. TF Dialect MLIR
# ============================================================
def tf_function_to_tf_mlir(fn):
    concrete_fn = fn.get_concrete_function(
        tf.TensorSpec([2], tf.float32),
        tf.TensorSpec([2], tf.float32),
    )
    return tf.mlir.experimental.convert_function(
        concrete_fn,
        pass_pipeline="tf-standard-pipeline",
        show_debug_info=False,
    )



tf_mlir = tf_function_to_tf_mlir(add_fn)
tf_mlir_path = f"{OUT_DIR}/tf_dialect.mlir"

with open(tf_mlir_path, "w") as f:
    f.write(tf_mlir)

print("=== TF DIALECT MLIR ===")
print(tf_mlir)


=== TF DIALECT MLIR ===
module attributes {tf.versions = {bad_consumers = [], min_consumer = 0 : i32, producer = 2288 : i32}} {
  func.func @__inference_add_fn_8(%arg0: tensor<2xf32> {tf._user_specified_name = "x"}, %arg1: tensor<2xf32> {tf._user_specified_name = "y"}) -> tensor<2xf32> attributes {allow_soft_placement = false, tf.entry_function = {control_outputs = "", inputs = "x,y", outputs = "identity_RetVal"}} {
    %0 = "tf.AddV2"(%arg0, %arg1) {device = ""} : (tensor<2xf32>, tensor<2xf32>) -> tensor<2xf32>
    %1 = "tf.Identity"(%0) {device = ""} : (tensor<2xf32>) -> tensor<2xf32>
    return %1 : tensor<2xf32>
  }
}



In [6]:
# ============================================================
# 3. STABLEHLO Dialect MLIR
# ============================================================
ir = add_fn.experimental_get_compiler_ir(x, y)(stage="stablehlo")
input_mlir = Path(OUT_DIR) / "00_input_stablehlo.mlir"
input_mlir.write_text(ir)

print("=== STABLEHLO MLIR ===")
print(input_mlir.read_text())

=== STABLEHLO MLIR ===
#loc1 = loc("XLA_Args")
module @a_inference_add_fn_8__.9 attributes {mhlo.cross_program_prefetches = [], mhlo.input_output_alias = [], mhlo.is_dynamic = false, mhlo.use_auto_spmd_partitioning = false} {
  func.func @main(%arg0: tensor<2xf32> loc("XLA_Args"), %arg1: tensor<2xf32> loc("XLA_Args")) -> tensor<2xf32> {
    %0 = stablehlo.reshape %arg0 : (tensor<2xf32>) -> tensor<2xf32> loc(#loc2)
    %1 = stablehlo.reshape %arg1 : (tensor<2xf32>) -> tensor<2xf32> loc(#loc3)
    %2 = stablehlo.add %0, %1 : tensor<2xf32> loc(#loc7)
    %3 = stablehlo.reshape %2 : (tensor<2xf32>) -> tensor<2xf32> loc(#loc6)
    return %3 : tensor<2xf32> loc(#loc)
  } loc(#loc)
} loc(#loc)
#loc = loc(unknown)
#loc2 = loc("reshape.3")
#loc3 = loc("reshape.4")
#loc4 = loc("add")
#loc5 = loc("/opt/conda/lib/python3.10/site-packages/tensorflow/python/framework/ops.py":1221:0)
#loc6 = loc("XLA_Retvals")
#loc7 = loc(fused[#loc4, #loc5])



In [7]:
out1 = Path(OUT_DIR) / "01_stablehlo_clean.mlir"

run([
    STABLEHLO_OPT,
    str(input_mlir),
    "--stablehlo-target-independent-optimization",
    "-o", str(out1),
])

print("=== STABLEHLO MLIR (Clean)===")
print(out1.read_text())


>> /workspace/stablehlo/build/bin/stablehlo-opt out/00_input_stablehlo.mlir --stablehlo-target-independent-optimization -o out/01_stablehlo_clean.mlir
=== STABLEHLO MLIR (Clean)===
module @a_inference_add_fn_8__.9 attributes {mhlo.cross_program_prefetches = [], mhlo.input_output_alias = [], mhlo.is_dynamic = false, mhlo.use_auto_spmd_partitioning = false} {
  func.func @main(%arg0: tensor<2xf32>, %arg1: tensor<2xf32>) -> tensor<2xf32> {
    %0 = stablehlo.add %arg0, %arg1 : tensor<2xf32>
    return %0 : tensor<2xf32>
  }
}




In [9]:
# ============================================================
# 4. StableHLO → Linalg (Tensor)
# ============================================================
out2_1 = Path(OUT_DIR) / "02_linalg_generic_tensor.mlir"

run([
    STABLEHLO_OPT,
    str(out1),
    "--stablehlo-legalize-to-linalg",
    "-linalg-fuse-elementwise-ops",  
    # "--linalg-specialize-generic-ops",  
    "--stablehlo-target-independent-optimization",
    "-o", str(out2_1),
])

print("=== GNERIC LINALG (TENSOR) MLIR ===")
print(out2_1.read_text())

>> /workspace/stablehlo/build/bin/stablehlo-opt out/01_stablehlo_clean.mlir --stablehlo-legalize-to-linalg -linalg-fuse-elementwise-ops --stablehlo-target-independent-optimization -o out/02_linalg_generic_tensor.mlir
=== GNERIC LINALG (TENSOR) MLIR ===
#map = affine_map<(d0) -> (d0)>
module @a_inference_add_fn_8__.9 attributes {mhlo.cross_program_prefetches = [], mhlo.input_output_alias = [], mhlo.is_dynamic = false, mhlo.use_auto_spmd_partitioning = false} {
  func.func @main(%arg0: tensor<2xf32>, %arg1: tensor<2xf32>) -> tensor<2xf32> {
    %0 = tensor.empty() : tensor<2xf32>
    %1 = linalg.generic {indexing_maps = [#map, #map, #map], iterator_types = ["parallel"]} ins(%arg0, %arg1 : tensor<2xf32>, tensor<2xf32>) outs(%0 : tensor<2xf32>) {
    ^bb0(%in: f32, %in_0: f32, %out: f32):
      %2 = arith.addf %in, %in_0 : f32
      linalg.yield %2 : f32
    } -> tensor<2xf32>
    return %1 : tensor<2xf32>
  }
}




In [10]:
# ============================================================
# 4.1 StableHLO → Linalg (Tensor)
# ============================================================
out2 = Path(OUT_DIR) / "02_linalg_specialized_tensor.mlir"

run([
    STABLEHLO_OPT,
    str(out2_1),
    "-linalg-fuse-elementwise-ops",  
    "--stablehlo-target-independent-optimization",
    "-o", str(out2),
])

print("=== LINALG (TENSOR) MLIR ===")
print(out2.read_text())

>> /workspace/stablehlo/build/bin/stablehlo-opt out/02_linalg_generic_tensor.mlir -linalg-fuse-elementwise-ops --stablehlo-target-independent-optimization -o out/02_linalg_specialized_tensor.mlir
=== LINALG (TENSOR) MLIR ===
#map = affine_map<(d0) -> (d0)>
module @a_inference_add_fn_8__.9 attributes {mhlo.cross_program_prefetches = [], mhlo.input_output_alias = [], mhlo.is_dynamic = false, mhlo.use_auto_spmd_partitioning = false} {
  func.func @main(%arg0: tensor<2xf32>, %arg1: tensor<2xf32>) -> tensor<2xf32> {
    %0 = tensor.empty() : tensor<2xf32>
    %1 = linalg.generic {indexing_maps = [#map, #map, #map], iterator_types = ["parallel"]} ins(%arg0, %arg1 : tensor<2xf32>, tensor<2xf32>) outs(%0 : tensor<2xf32>) {
    ^bb0(%in: f32, %in_0: f32, %out: f32):
      %2 = arith.addf %in, %in_0 : f32
      linalg.yield %2 : f32
    } -> tensor<2xf32>
    return %1 : tensor<2xf32>
  }
}




In [20]:
out6 = Path(OUT_DIR) / "06.mlir"

run([
    MLIR_OPT,
    str(out2),                      # bufferized input
    "-one-shot-bufferize=\"bufferize-function-boundaries\"",
    "-convert-bufferization-to-memref",
    "-convert-linalg-to-loops",
    # "-convert-vector-to-scf=full-unroll",
    # "-convert-index-to-llvm",
    # "-scf-for-loop-canonicalization",

    # "-convert-scf-to-cf",
    # "-convert-cf-to-llvm",

    # # "-convert-scf-to-cf",
    # "-lower-affine",
    # "-lower-vector-multi-reduction",


    # # # ---- memref / func / index ----
    # "-expand-strided-metadata",
    # "-finalize-memref-to-llvm",
    # "-convert-func-to-llvm",

    # # # ---- LLVM lowering ----
    # "-convert-arith-to-llvm",
    # "-convert-vector-to-llvm",
    # "-convert-math-to-llvm",
    
    # # # ---- cleanup ----
    # "-reconcile-unrealized-casts",

    "-o", str(out6),
])


print("=== LLVM DIALECT MLIR ===")
print(out6.read_text())


>> /LLVM-22.1.0-rc1-Linux-X64/bin/mlir-opt out/02_linalg_specialized_tensor.mlir -one-shot-bufferize="bufferize-function-boundaries" -convert-bufferization-to-memref -convert-linalg-to-loops -o out/06.mlir
=== LLVM DIALECT MLIR ===
module @a_inference_add_fn_8__.9 attributes {mhlo.cross_program_prefetches = [], mhlo.input_output_alias = [], mhlo.is_dynamic = false, mhlo.use_auto_spmd_partitioning = false} {
  func.func @main(%arg0: memref<2xf32, strided<[?], offset: ?>>, %arg1: memref<2xf32, strided<[?], offset: ?>>) -> memref<2xf32> {
    %c1 = arith.constant 1 : index
    %c2 = arith.constant 2 : index
    %c0 = arith.constant 0 : index
    %alloc = memref.alloc() {alignment = 64 : i64} : memref<2xf32>
    scf.for %arg2 = %c0 to %c2 step %c1 {
      %0 = memref.load %arg0[%arg2] : memref<2xf32, strided<[?], offset: ?>>
      %1 = memref.load %arg1[%arg2] : memref<2xf32, strided<[?], offset: ?>>
      %2 = arith.addf %0, %1 : f32
      memref.store %2, %alloc[%arg2] : memref<2xf32>
  

In [12]:
# ============================================================
# 2. LLVM dialect MLIR → LLVM IR (.ll)
# ============================================================
out_ll = Path(OUT_DIR) / "07.ll"

run([
    MLIR_TRANSLATE,
    "-mlir-to-llvmir",
    str(out6),
    "-o", str(out_ll),
])

print("=== LLVM IR (.ll) ===")
print(out_ll.read_text())


>> /LLVM-22.1.0-rc1-Linux-X64/bin/mlir-translate -mlir-to-llvmir out/06.mlir -o out/07.ll
=== LLVM IR (.ll) ===
; ModuleID = 'LLVMDialectModule'
source_filename = "LLVMDialectModule"

declare ptr @malloc(i64)

define { ptr, ptr, i64, [1 x i64], [1 x i64] } @main(ptr %0, ptr %1, i64 %2, i64 %3, i64 %4, ptr %5, ptr %6, i64 %7, i64 %8, i64 %9) {
  %11 = call ptr @malloc(i64 72)
  %12 = ptrtoint ptr %11 to i64
  %13 = add i64 %12, 63
  %14 = urem i64 %13, 64
  %15 = sub i64 %13, %14
  %16 = inttoptr i64 %15 to ptr
  %17 = insertvalue { ptr, ptr, i64, [1 x i64], [1 x i64] } poison, ptr %11, 0
  %18 = insertvalue { ptr, ptr, i64, [1 x i64], [1 x i64] } %17, ptr %16, 1
  %19 = insertvalue { ptr, ptr, i64, [1 x i64], [1 x i64] } %18, i64 0, 2
  %20 = insertvalue { ptr, ptr, i64, [1 x i64], [1 x i64] } %19, i64 2, 3, 0
  %21 = insertvalue { ptr, ptr, i64, [1 x i64], [1 x i64] } %20, i64 1, 4, 0
  br label %22

22:                                               ; preds = %25, %10
  %23 = phi i64 

In [13]:
# ============================================================
# 3. LLVM IR → RISC-V Assembly
# ============================================================

In [14]:
out_s = Path(OUT_DIR) / "06_riscv.s"

run([
    "/riscv-llvm/bin/llc",
    "-relocation-model=pic",
    "-march=riscv64",
    "-O3",
    "--stack-size-section",
    "-mattr=+m,+f,+d,+a,+c,+v,+xsfvcp,zvl256b",
    "-O2",
    str(out_ll),
    "-o", str(out_s),
])

print("=== RISC-V ASM ===")
print(out_s.read_text())


>> /riscv-llvm/bin/llc -relocation-model=pic -march=riscv64 -O3 --stack-size-section -mattr=+m,+f,+d,+a,+c,+v,+xsfvcp,zvl256b -O2 out/07.ll -o out/06_riscv.s
=== RISC-V ASM ===
	.text
	.attribute	4, 16
	.attribute	5, "rv64i2p1_m2p0_a2p1_f2p2_d2p2_c2p0_v1p0_zicsr2p0_zmmul1p0_zve32f1p0_zve32x1p0_zve64d1p0_zve64f1p0_zve64x1p0_zvl128b1p0_zvl256b1p0_zvl32b1p0_zvl64b1p0_xsfvcp1p0"
	.file	"LLVMDialectModule"
	.globl	main                            # -- Begin function main
	.p2align	1
	.type	main,@function
main:                                   # @main
.Lfunc_begin0:
	.cfi_startproc
# %bb.0:
	addi	sp, sp, -64
	.cfi_def_cfa_offset 64
	sd	ra, 56(sp)                      # 8-byte Folded Spill
	sd	s0, 48(sp)                      # 8-byte Folded Spill
	sd	s1, 40(sp)                      # 8-byte Folded Spill
	sd	s2, 32(sp)                      # 8-byte Folded Spill
	sd	s3, 24(sp)                      # 8-byte Folded Spill
	sd	s4, 16(sp)                      # 8-byte Folded Spill
	sd	s5, 8(sp)     

In [15]:
import torch, os, sys, subprocess, re, io
from pathlib import Path
from contextlib import redirect_stdout

f = io.StringIO()
with redirect_stdout(f):
    base_dir = os.environ.get("TORCHSIM_DIR", "/workspace/PyTorchSim")
    sys.path.append(base_dir)
    from Scheduler.scheduler import PyTorchSimRunner
    device = PyTorchSimRunner.setup_device().custom_device()

    a = torch.tensor([1.0], dtype=torch.float32, device=device)
    b = torch.tensor([3.0, 4.0], dtype=torch.float32, device=device)

    def add(x, y):
        return torch.add(x, y)

    opt_fn = torch.compile(dynamic=False)(add)
    _ = opt_fn(a, b)

stdout = f.getvalue()
m = re.search(r"Wrapper Codegen Path = (.+)", stdout)
wrapper_path = Path(m.group(1).strip())
code = wrapper_path.read_text()
mlir_match = re.search(r"custom_async_compile\.mlir\(\s*'''(.*?)'''\s*,?",code,re.DOTALL,)
print(mlir_match.group(1))


/opt/conda/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Using /root/.cache/torch_extensions/py310_cu121 as PyTorch extensions root...
Emitting ninja build file /root/.cache/torch_extensions/py310_cu121/npu/build.ninja...
Building extension module npu...
Allowing ninja to set a default number of workers... (overridable by setting the environment variable MAX_JOBS=N)
Loading extension module npu...


ninja: no work to do.
memref.global @buf0_spad : memref<2xf32, 1>
memref.global @buf1_spad : memref<2xf32, 1>
memref.global @buf2_spad : memref<2xf32, 1>
func.func @kernel(%in_ptr0: memref<1xf32>,
                       %in_ptr1: memref<2xf32>,
                       %out_ptr0: memref<2xf32>)
{
    %const0 = arith.constant 0 : index
    %const1 = arith.constant 2 : index
    %const2 = arith.constant 3 : index
    %alloc0 = memref.alloc() : memref<1xi32> // 0
    %alloc1 = memref.alloc() : memref<1xi32> // 1
    %alloc2 = memref.alloc() : memref<1xi32> // 2
    %spad0 = memref.get_global @buf0_spad : memref<2xf32, 1>
    %spad1 = memref.get_global @buf1_spad : memref<2xf32, 1>
    %spad2 = memref.get_global @buf2_spad : memref<2xf32, 1>
    affine.for %index0 = 0 to 2 step 2
    {
        memref.dma_start %in_ptr0[%const0], %spad0[%const0], %const1, %alloc0[%const0], %const0, %const1 : memref<1xf32>, memref<2xf32, 1>, memref<1xi32> {dram_stride=[0], sram_stride=[1], padding=0}
        m